In [ ]:
!unzip stealthwatch_training_bundle.zip

Archive:  stealthwatch_training_bundle.zip
   creating: src/eval/
  inflating: src/cicids_loader.py    
  inflating: src/data_loader.py      
  inflating: src/preprocess.py       
  inflating: src/beaconer/beaconer.py  
  inflating: src/beaconer/listener.py  
  inflating: src/beaconer/__pycache__/beaconer.cpython-313.pyc  
  inflating: src/classifier/ocsvm.py  
  inflating: src/classifier/__pycache__/ocsvm.cpython-313.pyc  
  inflating: src/entropy/feature_extractor.py  
  inflating: src/entropy/__pycache__/feature_extractor.cpython-313.pyc  
  inflating: src/tap/pcap_tap.py     
  inflating: src/theory/entropy_bound.py  
  inflating: requirements.txt        


In [ ]:
!pip install antropy pandas==2.2.2 scikit-learn numpy scipy
import antropy
print(f'Antropy version: {antropy.__version__}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 112.7 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.3
    Uninstalling pandas-3.0.3:
      Successfully uninstalled pandas-3.0.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
Antropy version: 0.2.2


In [ ]:
!mkdir -p data/raw/CTU-13 data/raw/CICIDS2017 data/processed

In [ ]:
# Download Scenario 1 Labeled NetFlow from the updated repository
!wget -O data/raw/CTU-13/scenario1.labeled https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-42/detailed-bidirectional-flow-labels/capture20110810.binetflow

--2026-05-16 13:47:11--  https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-42/detailed-bidirectional-flow-labels/capture20110810.binetflow
Resolving mcfp.felk.cvut.cz (mcfp.felk.cvut.cz)... 147.32.82.194
Connecting to mcfp.felk.cvut.cz (mcfp.felk.cvut.cz)|147.32.82.194|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 386629271 (369M)
Saving to: ‘data/raw/CTU-13/scenario1.labeled’

data/raw/CTU-13/sce 100%[===================>] 368.72M  7.86MB/s    in 55s     

2026-05-16 13:48:07 (6.73 MB/s) - ‘data/raw/CTU-13/scenario1.labeled’ saved [386629271/386629271]



In [ ]:
import sys
import os

# Optimize the script with vectorized grouping to handle large datasets
new_script_content = """
import numpy as np
import pandas as pd
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score
import joblib
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))
from src.entropy.feature_extractor import EntropyFeatureExtractor

def optimized_load_and_group(path):
    print("Reading CSV...")
    df = pd.read_csv(path)
    df['is_malicious'] = df['Label'].str.contains('Botnet', na=False).astype(int)
    df['timestamp'] = pd.to_datetime(df['StartTime']).astype('int64') // 10**9

    print(f"Processing {len(df)} flows...")
    # Group by Host and Connection, then collect timestamps into lists
    grouped = df.sort_values('timestamp').groupby(['SrcAddr', 'DstAddr', 'Dport'])

    # Filter connection groups with enough packets (minimum 20)
    agg_df = grouped.agg(
        deltas=('timestamp', lambda x: np.diff(x.values) if len(x) >= 20 else None),
        label=('is_malicious', 'max')
    ).dropna()

    return agg_df

class StealthwatchClassifier:
    def __init__(self, nu=0.05):
        self.nu = nu
        self.ocsvm = OneClassSVM(nu=nu, kernel='rbf', gamma='scale')
        self.scaler = StandardScaler()

    def fit(self, X):
        X_scaled = self.scaler.fit_transform(X)
        self.ocsvm.fit(X_scaled)

    def evaluate(self, X_test, y_test, label='Test'):
        X_scaled = self.scaler.transform(X_test)
        y_pred = (self.ocsvm.predict(X_scaled) == -1).astype(int)
        scores = self.ocsvm.decision_function(X_scaled)
        print(f'=== {label} ===')
        print(f'  F1: {f1_score(y_test, y_pred):.4f}')
        print(f'  AUC: {roc_auc_score(y_test, -scores):.4f}')

if __name__ == "__main__":
    data_path = 'data/raw/CTU-13/scenario1.labeled'
    agg_df = optimized_load_and_group(data_path)

    extractor = EntropyFeatureExtractor(window=50)
    benign_feats, malicious_feats = [], []

    print("Extracting entropy features...")
    for _, row in agg_df.iterrows():
        deltas = row['deltas']
        deltas = deltas[deltas > 0]
        if len(deltas) >= 15:
            f = extractor.compute_features(deltas)
            if f is not None:
                if row['label'] == 1: malicious_feats.append(f)
                else: benign_feats.append(f)

    print(f"Extracted Benign: {len(benign_feats)}, Malicious: {len(malicious_feats)}")

    X_benign = np.array(benign_feats)
    split = int(0.8 * len(X_benign))
    X_train, X_test_benign = X_benign[:split], X_benign[split:]

    X_test = np.vstack([X_test_benign, malicious_feats])
    y_test = np.hstack([np.zeros(len(X_test_benign)), np.ones(len(malicious_feats))])

    clf = StealthwatchClassifier()
    clf.fit(X_train)
    clf.evaluate(X_test, y_test, label='CTU-13 Real Data (Optimized)')

    os.makedirs('experiments', exist_ok=True)
    joblib.dump(clf, 'experiments/stealthwatch_model.pkl')
    print("Model saved to experiments/stealthwatch_model.pkl")
"""

with open('src/classifier/ocsvm.py', 'w') as f:
    f.write(new_script_content.strip())

!{sys.executable} src/classifier/ocsvm.py

Reading CSV...
Processing 2824636 flows...
Extracting entropy features...
Extracted Benign: 1171, Malicious: 94
=== CTU-13 Real Data (Optimized) ===
  F1: 0.1111
  AUC: 0.8569
Model saved to experiments/stealthwatch_model.pkl


In [ ]:
import os

# List of all 13 scenario URLs for the labeled binetflow files
scenarios = {
    1: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-42/detailed-bidirectional-flow-labels/capture20110810.binetflow',
    2: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-43/detailed-bidirectional-flow-labels/capture20110811.binetflow',
    3: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-44/detailed-bidirectional-flow-labels/capture20110812.binetflow',
    4: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-45/detailed-bidirectional-flow-labels/capture20110815.binetflow',
    5: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-46/detailed-bidirectional-flow-labels/capture20110815-2.binetflow',
    6: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-47/detailed-bidirectional-flow-labels/capture20110816.binetflow',
    7: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-48/detailed-bidirectional-flow-labels/capture20110816-2.binetflow',
    8: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-49/detailed-bidirectional-flow-labels/capture20110816-3.binetflow',
    9: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-50/detailed-bidirectional-flow-labels/capture20110817.binetflow',
    10: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-51/detailed-bidirectional-flow-labels/capture20110818.binetflow',
    11: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-52/detailed-bidirectional-flow-labels/capture20110818-2.binetflow',
    12: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-53/detailed-bidirectional-flow-labels/capture20110819.binetflow',
    13: 'https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-54/detailed-bidirectional-flow-labels/capture20110815-3.binetflow'
}

print("Downloading all 13 scenarios...")
for s_id, url in scenarios.items():
    target_path = f'data/raw/CTU-13/scenario{s_id}.labeled'
    if not os.path.exists(target_path):
        print(f"Downloading Scenario {s_id}...")
        os.system(f'wget -q -O {target_path} {url}')
    else:
        print(f"Scenario {s_id} already exists.")

Scenario 1 already exists.


In [ ]:
import sys
import os

# Pivot to Supervised Learning to verify feature relevance
supervised_script = """
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, roc_auc_score
import joblib
import os
import glob
from scipy.stats import entropy

def extract_features(group):
    deltas = np.diff(group['timestamp'].values)
    deltas = deltas[deltas > 0]
    if len(deltas) < 5: return None

    return [
        entropy(np.histogram(deltas, bins=10, density=True)[0] + 1e-9),
        np.mean(deltas),
        np.var(deltas),
        group['Dur'].mean(),
        group['TotPkts'].sum(),
        group['TotBytes'].sum()
    ]

def load_all_scenarios(file_paths):
    rows = []
    for path in file_paths:
        print(f"Processing {path}...")
        df = pd.read_csv(path)
        df['is_malicious'] = df['Label'].str.contains('Botnet', na=False).astype(int)
        df['timestamp'] = pd.to_datetime(df['StartTime']).astype('int64') // 10**9

        for _, group in df.groupby(['SrcAddr', 'DstAddr', 'Dport']):
            if len(group) >= 10:
                f = extract_features(group)
                if f:
                    rows.append(f + [group['is_malicious'].max()])
    return pd.DataFrame(rows, columns=['ent', 'iat_mean', 'iat_var', 'dur', 'pkts', 'bytes', 'label'])

if __name__ == '__main__':
    files = sorted(glob.glob('data/raw/CTU-13/*.labeled'))
    data = load_all_scenarios(files)

    X = data.drop('label', axis=1)
    y = data['label']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    probs = clf.predict_proba(X_test)[:, 1]

    print("=== Supervised Random Forest Results ===")
    print(classification_report(y_test, y_pred))
    print(f"  AUC: {roc_auc_score(y_test, probs):.4f}")

    joblib.dump(clf, 'experiments/stealthwatch_supervised.pkl')
    print("Supervised model saved to experiments/stealthwatch_supervised.pkl")
"""

with open('src/classifier/train_supervised.py', 'w') as f:
    f.write(supervised_script.strip())

!{sys.executable} src/classifier/train_supervised.py

Processing data/raw/CTU-13/scenario1.labeled...
Processing data/raw/CTU-13/scenario10.labeled...
Processing data/raw/CTU-13/scenario11.labeled...
Processing data/raw/CTU-13/scenario12.labeled...
Processing data/raw/CTU-13/scenario13.labeled...
Processing data/raw/CTU-13/scenario2.labeled...
Processing data/raw/CTU-13/scenario3.labeled...
Processing data/raw/CTU-13/scenario4.labeled...
Processing data/raw/CTU-13/scenario5.labeled...
Processing data/raw/CTU-13/scenario6.labeled...
Processing data/raw/CTU-13/scenario7.labeled...
Processing data/raw/CTU-13/scenario8.labeled...
Processing data/raw/CTU-13/scenario9.labeled...
=== Supervised Random Forest Results ===
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     19704
           1       0.91      0.52      0.66       639

    accuracy                           0.98     20343
   macro avg       0.95      0.76      0.83     20343
weighted avg       0.98      0.98      0.98     20343

  AU